# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (Kakfa producer)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

kafka_connector = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
su = SparkUtils("Example Kafka", 
                "spark://spark-master:7077",
                spark_packages=kafka_connector)
su.spark


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ca87f9f7-12c1-4468-9ca6-5f59132c2319;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0

# Create a data stream from a Kafka topic

In [3]:
# Create the remote connection
kafka_df = su.spark.readStream \
            .format("kafka") \
            .option("kafka.bootstrap.servers", "kafka:9093") \
            .option("subscribe", "kafka-spark-example") \
            .load()

kafka_df.printSchema()

# Transform binary data to string
df_input = kafka_df.selectExpr("CAST(value AS STRING)")

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

words = df_input.select(F.explode(F.split(df_input.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query_a = (word_count.writeStream
            .trigger(processingTime='2 second')
            .outputMode("complete")
            .format("console")
            .option("checkpointLocation", checkpoint_path)
            .start())

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

   Press Ctrl+C to stop.



26/04/14 01:05:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
|exit|    1|
+----+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+------------+-----+
|        word|count|
+------------+-----+
|        exit|    2|
|kafka-python|    1|
|     install|    2|
|         pip|    2|
+------------+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
| exit|    1|
|/exit|    1|
+-----+-----+

-------------------------------------------
Batch: 4
-------------------------------------------
+------------+-----+
|        word|count|
+------------+-----+
|        exit|    2|
|kafka-python|    1|
|     install|    2|
|         pip|    2|
|       /exit|    1|
+------------+-----+



26/04/14 01:09:18 ERROR MicroBatchExecution: Query [id = a7981eda-b172-4663-9962-c5fd358dd6e0, runId = fa01f3ac-d037-4db7-be44-2c801a64bb3d] terminated with error
org.apache.spark.SparkException: [CONCURRENT_STREAM_LOG_UPDATE] Concurrent update to the log. Multiple streaming jobs detected for 3.
Please make sure only one streaming job runs on a specific checkpoint location at a time. SQLSTATE: 40000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.concurrentStreamLogUpdate(QueryExecutionErrors.scala:1227)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution.markMicroBatchStart(MicroBatchExecution.scala:912)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution.$anonfun$constructNextBatch$15(MicroBatchExecution.scala:666)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.sql.execution.streaming.ProgressContext.reportTimeTaken(ProgressReporter.scala:186)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecu

StreamingQueryException: [STREAM_FAILED] Query [id = a7981eda-b172-4663-9962-c5fd358dd6e0, runId = fa01f3ac-d037-4db7-be44-2c801a64bb3d] terminated with exception: [CONCURRENT_STREAM_LOG_UPDATE] Concurrent update to the log. Multiple streaming jobs detected for 3.
Please make sure only one streaming job runs on a specific checkpoint location at a time. SQLSTATE: 40000 SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = a7981eda-b172-4663-9962-c5fd358dd6e0, runId = fa01f3ac-d037-4db7-be44-2c801a64bb3d]
Current Committed Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":6}}}
Current Available Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":7}}}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSource org.apache.spark.sql.execution.streaming.ConsoleTable$@183690b9, a7981eda-b172-4663-9962-c5fd358dd6e0, [checkpointLocation=/opt/spark/work-dir/checkpoints/logs_checkpoint], Complete
+- ~Aggrega

## Custom producer

### Create `server-logs` topic

```
    docker exec -it <Kafka container ID> \
     /opt/kafka/bin/kafka-topics.sh \
      --create --zookeeper zookeeper:2181 \
      --replication-factor 1 --partitions 1 \
      --topic server-logs
```

### Run the producer

```
    docker exec -it <Spark-Notebook container ID> /bin/bash
    # cd src/producers/
    # python3 kafka_producer.py --broker kafka:9093 --topic server-logs --records 20
```

### Run the consumer code

In [5]:
# Create the remote connection
server_logs_df = (su.spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "kafka:9093")
            .option("subscribe", "server-logs")
            .load())

# Transform binary data to string
logs_df = server_logs_df.selectExpr("CAST(value AS STRING)")

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("value"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "value")
    .filter(F.col("timestamp").isNotNull())
)

# Write stream in the destination
output_path = "/opt/spark/work-dir/data/streaming/output/"
query_events = (
    parsed_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("truncate", False)
    .option("checkpointLocation", checkpoint_path)
    .option("path", output_path)
    .partitionBy("server", "level")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

   Press Ctrl+C to stop.



26/04/14 01:10:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


StreamingQueryException: [STREAM_FAILED] Query [id = a7981eda-b172-4663-9962-c5fd358dd6e0, runId = fa01f3ac-d037-4db7-be44-2c801a64bb3d] terminated with exception: [CONCURRENT_STREAM_LOG_UPDATE] Concurrent update to the log. Multiple streaming jobs detected for 3.
Please make sure only one streaming job runs on a specific checkpoint location at a time. SQLSTATE: 40000 SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = a7981eda-b172-4663-9962-c5fd358dd6e0, runId = fa01f3ac-d037-4db7-be44-2c801a64bb3d]
Current Committed Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":6}}}
Current Available Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":7}}}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSource org.apache.spark.sql.execution.streaming.ConsoleTable$@183690b9, a7981eda-b172-4663-9962-c5fd358dd6e0, [checkpointLocation=/opt/spark/work-dir/checkpoints/logs_checkpoint], Complete
+- ~Aggrega

In [10]:
su.spark.stop()